#### Section 1 - Dependancy install, library import, client substation, create constants
###### Step 1 - Install dependancies
###### Step 2 - Import libraries and setup clients
###### Step 3 - Create names for S3 and S3 Vectors buckets, index name, and media
###### Step 4 - Create S3 bucket for video and embeddings, S3 Vectors bucket and index for embeddings and queries


In [1]:
# Section 1 - Step 1 - Install dependencies

!pip install boto3==1.39.7
!pip install requests
!pip install numpy

In [2]:
# Section 1 - Step 2 - Import libraries and initialize clients

import boto3
import json
import os
import uuid
import requests
import re
import time
import numpy as np
from datetime import datetime
from IPython.display import Video, HTML

# Initialize Bedrock Runtime client
bedrock_runtime = boto3.client('bedrock-runtime', region_name='us-east-1')
s3 = boto3.client("s3", region_name = 'us-east-1')
s3_vectors = boto3.client("s3vectors", region_name = 'us-east-1')

# Get AWS account ID for bucket owner
sts = boto3.client('sts')
account_id = sts.get_caller_identity()['Account']


In [3]:
# Section 1 - Step 3 - Create constants

# Generate unique identifiers for S3 buckets and create names 
bucket_uid = str(uuid.uuid4())[:8] 
vector_uid = str(uuid.uuid4())[:8]
s3_bucket_name = f"test-video-demo-{bucket_uid}"
vector_bucket_name = f"test-vectors-{vector_uid}"

# Index name for and topk results for S3 Vectors
indexName = 'media'
top_k = 5  # Number of results to return

test_video_url = "http://ftp.nluug.nl/pub/graphics/blender/demo/movies/ToS/tears_of_steel_720p.mov"
test_video_name="tears-of-steel.mov"

print(f"S3 Bucket Name: {s3_bucket_name}")
print(f"Vector Bucket Name: {vector_bucket_name}")


S3 Bucket Name: test-video-demo-41e28b56
Vector Bucket Name: test-vectors-34294b48


In [4]:
# Section 1 - Step 4 - Create S3 bucket, S3 Vectors bucket, and index

try:
    s3.create_bucket(Bucket=s3_bucket_name)
    print(f"Created S3 bucket: {s3_bucket_name}")
except Exception as e:
    print(f"Error creating S3 bucket: {e}")

# Create S3 Vector bucket
try:
    s3_vectors.create_vector_bucket(
        vectorBucketName=vector_bucket_name
    )
    print(f"Created S3 Vector bucket: {vector_bucket_name}")
except Exception as e:
    print(f"Error creating S3 Vector bucket: {e}")

# Create vector index
try:
    response = s3_vectors.create_index(
        vectorBucketName=vector_bucket_name,
        indexName=indexName,
        dataType='float32',
        dimension=1024,
        distanceMetric="cosine"
    )
    print(f"Created vector index: {indexName} in bucket: {vector_bucket_name}")
except Exception as e:
    print(f"Error creating vector index: {e}")

Created S3 bucket: test-video-demo-41e28b56
Created S3 Vector bucket: test-vectors-34294b48
Created vector index: media in bucket: test-vectors-34294b48


##### Section 2 - Video download, embedding creation, and embedding insertion
###### Step 1 - Video download and upload to S3 bucket (This can take several minutes depending on your Internet connection)
###### Step 2 - TwelveLabs Marengo 2.7 on Bedrock inference and embedding generation
###### Step 3 - Insertion of embeddings into S3 Vectors index



In [8]:
# Section 2 - Step 1 - Video download and upload to S3 bucket

def download_video(url, local_filename):
    # Download video from URL
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(local_filename, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
    return local_filename

def upload_to_s3(local_filename, bucket_name, object_key):
    # Upload file to AWS S3
    s3 = boto3.client('s3')
    s3.upload_file(local_filename, bucket_name, object_key)
    print(f"Uploaded {local_filename} to s3://{bucket_name}/{object_key}")

if __name__ == "__main__":
    video_url = test_video_url
    bucket_name = s3_bucket_name
    object_key = test_video_name

print("Downloading video... (This may take multiple minutes depending on your internet speed)")
download_video(video_url, test_video_name)
print("Uploading to S3...(This may take multiple minutes depending on your internet speed)")
upload_to_s3(test_video_name, bucket_name, object_key)


Uploading to S3...(This may take multiple minutes depending on your internet speed)
Uploaded tears-of-steel.mov to s3://test-video-demo-081df67e/tears-of-steel.mov


In [13]:
# Section 2 - Step 2 - TwelveLabs Marengo 2.7 on Bedrock inference and embedding generation

try:
    response = bedrock_runtime.start_async_invoke(
        modelId="twelvelabs.marengo-embed-2-7-v1:0",
        modelInput={
            "inputType": "video",
            "mediaSource": {
                    "s3Location": {
                        "uri": f"s3://{s3_bucket_name}/{test_video_name}",
                        "bucketOwner": account_id
                    }
                }
        },
        outputDataConfig={
            "s3OutputDataConfig": {
                "s3Uri": f"s3://{s3_bucket_name}/videoEmbedding"
            }
        }
    )
    
    video_invocation_arn = response['invocationArn']
    print(f"Started video embedding async invoke with ARN: {video_invocation_arn}")
    
    # Extract the UID from the video invocation ARN
    video_uid_match = re.search(r'/([^/]+)$', video_invocation_arn)
    if video_uid_match:
        video_uid = video_uid_match.group(1)
        embedding_video_response_location = f"videoEmbedding/{video_uid_match.group(1)}"
        print(f"Extracted video UID: {video_uid}")
        
        # Wait for video embedding to complete
        max_wait_time = 3000  # 50 minutes
        wait_interval = 10   # 10 seconds
        waited_time = 0

        print("Waiting for video embedding to complete...")

        while waited_time < max_wait_time:
            try:
                # Check for response file
                response = s3.list_objects_v2(
                    Bucket=s3_bucket_name,
                    Prefix=embedding_video_response_location
                )
                
                if 'Contents' in response:
                    response_files = [obj for obj in response['Contents'] if obj['Key'].endswith('output.json')]
                    
                    if response_files:
                        # Download and parse the response
                        response_key = response_files[0]['Key']
                        response_obj = s3.get_object(Bucket=s3_bucket_name, Key=response_key)
                        response_content = response_obj['Body'].read().decode('utf-8')
                        response_data = json.loads(response_content)
                        
                        # Extract embeddings data
                        if 'data' in response_data and len(response_data['data']) > 0:
                            embeddings_data = response_data['data']
                            print(f"Video embedding completed successfully - found {len(embeddings_data)} embeddings")
                            break
                        else:
                            print("No embedding data found in response")
                    else:
                        print(f"Waiting... ({waited_time}s elapsed)")
                else:
                    print(f"Waiting... ({waited_time}s elapsed)")
                    
            except Exception as e:
                print(f"Error checking for response: {e}")
            
            time.sleep(wait_interval)
            waited_time += wait_interval

        if waited_time >= max_wait_time:
            print("Timeout waiting for video embedding to complete")
            embeddings_data = None
        
    else:
        print("Could not extract UID from video invocation ARN")
        embeddings_data = None
        
except Exception as e:
    print(f"Error starting video embedding async invoke: {e}")
    embeddings_data = None

Started video embedding async invoke with ARN: arn:aws:bedrock:us-east-1:177456485324:async-invoke/dpwimwi05q8p
Extracted video UID: dpwimwi05q8p
Waiting for video embedding to complete...
Waiting... (0s elapsed)
Waiting... (10s elapsed)
Waiting... (20s elapsed)
Waiting... (30s elapsed)
Waiting... (40s elapsed)
Waiting... (50s elapsed)
Video embedding completed successfully - found 354 embeddings


In [21]:
# Step 3 - Insertion of embeddings into S3 Vectors index

if embeddings_data is not None and len(embeddings_data) > 0:
    print(f"\nSection 2: Processing {len(embeddings_data)} embeddings for S3 Vector insertion...")
    
    # Prepare vectors for S3 Vector insertion
    vectors_to_insert = []
    
    for i, embedding_obj in enumerate(embeddings_data):
        # Extract required fields
        embedding_vector = embedding_obj.get('embedding', [])
        start_sec = embedding_obj.get('startSec', 0.0)
        end_sec = embedding_obj.get('endSec', 0.0)
        embedding_option = embedding_obj.get('embeddingOption', 'unknown')
        
        # Ensure embedding is float32 format
        embedding_float32 = np.array(embedding_vector, dtype=np.float32).tolist()
        
        # Create unique key for this vector
        vector_key = f"{test_video_name}_segment_{i:04d}_{start_sec:.2f}-{end_sec:.2f}_{embedding_option}"
        
        # Prepare vector data structure for S3 Vectors
        vector_data = {
            'key': vector_key,
            'data': {
                'float32': embedding_float32
            },
            'metadata': {
                'video_uri': f"s3://{s3_bucket_name}/{test_video_name}",
                'start_sec': start_sec,
                'end_sec': end_sec,
                'embedding_option': embedding_option,
            }
        }
        
        vectors_to_insert.append(vector_data)
        
        # Only show details for first 10 vectors to avoid log clutter
        if i < 10:
            print(f"Prepared vector {i}: {vector_key} (dim: {len(embedding_float32)})")
        elif i == 10:
            print(f"... (showing first 10 vectors, continuing to prepare remaining {len(embeddings_data) - 10} vectors)")
    
    print(f"\n✅ Prepared all {len(vectors_to_insert)} vectors for insertion")
    
    # Insert vectors into S3 Vector index
    try:
        print(f"\nInserting {len(vectors_to_insert)} vectors into S3 Vector index...")
        
        # Insert vectors (may need to batch if too many)
        batch_size = 100  # Adjust based on service limits
        total_start_time = time.time()
        
        for i in range(0, len(vectors_to_insert), batch_size):
            batch = vectors_to_insert[i:i + batch_size]
            batch_number = i//batch_size + 1
            
            print(f"🔄 Starting batch {batch_number} insertion ({len(batch)} vectors)...")
            
            # Time each batch insertion
            batch_start_time = time.time()
            
            response = s3_vectors.put_vectors(
                vectorBucketName=vector_bucket_name,
                indexName=indexName,
                vectors=batch
            )
            
            batch_end_time = time.time()
            batch_duration_ms = (batch_end_time - batch_start_time) * 1000
            
            print(f"✅ Batch {batch_number} completed ({len(batch)} vectors) - {batch_duration_ms:.1f} ms")
            print()  # Add blank line for better readability
        
        total_end_time = time.time()
        total_duration_ms = (total_end_time - total_start_time) * 1000
        
        print(f"🎉 All {len(vectors_to_insert)} vectors inserted successfully!")
        print(f"⏱️  Total insertion time: {total_duration_ms:.1f} milliseconds")
        
    except Exception as e:
        print(f"Error inserting vectors into S3 Vector index: {e}")
        print("Vector insertion failed")
        
else:
    print("Section 2: No embeddings data available for S3 Vector insertion")   


Section 2: Processing 354 embeddings for S3 Vector insertion...
Prepared vector 0: tears-of-steel.mov_segment_0000_0.00-8.00_visual-text (dim: 1024)
Prepared vector 1: tears-of-steel.mov_segment_0001_8.00-12.60_visual-text (dim: 1024)
Prepared vector 2: tears-of-steel.mov_segment_0002_12.60-17.50_visual-text (dim: 1024)
Prepared vector 3: tears-of-steel.mov_segment_0003_17.50-24.10_visual-text (dim: 1024)
Prepared vector 4: tears-of-steel.mov_segment_0004_24.10-29.20_visual-text (dim: 1024)
Prepared vector 5: tears-of-steel.mov_segment_0005_29.20-34.30_visual-text (dim: 1024)
Prepared vector 6: tears-of-steel.mov_segment_0006_34.30-39.40_visual-text (dim: 1024)
Prepared vector 7: tears-of-steel.mov_segment_0007_39.40-44.70_visual-text (dim: 1024)
Prepared vector 8: tears-of-steel.mov_segment_0008_44.70-48.90_visual-text (dim: 1024)
Prepared vector 9: tears-of-steel.mov_segment_0009_48.90-55.80_visual-text (dim: 1024)
... (showing first 10 vectors, continuing to prepare remaining 344 v

##### Section 3 - Text embedding creation, vector embedding search, and media playback
###### Step 1 - Creation of vector embedding for text search phrase
###### Step 2 - Query S3 Vectors for topk nearest neighbors
###### Step 3 - Play video back at the start of the time code from the nearest neighbor

In [29]:
# Section 3 - Step 1 - Creation of vector embedding for text search phrase

text_search_phrase = "a person fighting a robot"

# Start the async invoke for text embedding
try:
    # Capture start time
    start_time = time.time()
    start_datetime = datetime.now()
    print(f"🕐 Starting Bedrock async invoke at: {start_datetime.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]}")
    
    response = bedrock_runtime.start_async_invoke(
        modelId="twelvelabs.marengo-embed-2-7-v1:0",
        modelInput={
            "inputType": "text",
            "inputText": text_search_phrase
        },
        outputDataConfig={
            "s3OutputDataConfig": {
                "s3Uri": f"s3://{s3_bucket_name}/textEmbedding"
            }
        }
    )
    
    text_invocation_arn = response['invocationArn']
    print(f"Started text embedding async invoke with ARN: {text_invocation_arn}")
    
except Exception as e:
    print(f"Error starting text embedding async invoke: {e}")

# Extract the UID from the text invocation ARN
text_uid_match = re.search(r'/([^/]+)$', text_invocation_arn)
if text_uid_match:
    text_uid = text_uid_match.group(1)
    embedding_text_response_location = f"textEmbedding/{text_uid_match.group(1)}"
    print(f"Extracted text UID: {text_uid}")
else:
    print("Could not extract UID from text invocation ARN")
    text_uid = None

# Wait for text embedding to complete and then retrieve the result
max_wait_time = 150  # 5 minutes
wait_interval = 1   # 1 second
waited_time = 0

print("Waiting for text embedding to complete...")

while waited_time < max_wait_time:
    try:
        # List objects in the text embedding response location
        response = s3.list_objects_v2(
            Bucket=s3_bucket_name,
            Prefix=embedding_text_response_location
        )
        
        if 'Contents' in response:
            # Look for the response file
            response_files = [obj for obj in response['Contents'] if obj['Key'].endswith('output.json')]
            
            if response_files:
                # Capture end time when output.json is found
                end_time = time.time()
                end_datetime = datetime.now()
                total_duration = end_time - start_time
                
                print(f"🎯 Found output.json at: {end_datetime.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]}")
                print(f"⏱️  TOTAL PROCESSING TIME: {total_duration:.2f} seconds ({total_duration/60:.2f} minutes)")
                print("=" * 60)

                # Get the response file
                response_key = response_files[0]['Key']
                print(f"Found response file: {response_key}")
                
                # Download and parse the response
                response_obj = s3.get_object(Bucket=s3_bucket_name, Key=response_key)
                response_content = response_obj['Body'].read().decode('utf-8')
                response_data = json.loads(response_content)
                
                print("Response content preview:")
                print(json.dumps(response_data, indent=2)[:500] + "..." if len(json.dumps(response_data, indent=2)) > 500 else json.dumps(response_data, indent=2))
                
                # Extract the embedding vector
                if 'data' in response_data:
                    text_embedding_vector = response_data['data'][0]['embedding']
                    print(f"\nExtracted text embedding vector (length: {len(text_embedding_vector)})")
                    print(f"First 10 values: {text_embedding_vector[:10]}")
                    break
                else:
                    print("No 'embedding' field found in response")
            else:
                print(f"No response files found yet, waiting... ({waited_time}s elapsed)")
        else:
            print(f"No objects found yet, waiting... ({waited_time}s elapsed)")
            
    except Exception as e:
        print(f"Error checking for response: {e}")
    
    time.sleep(wait_interval)
    waited_time += wait_interval

if waited_time >= max_wait_time:
    print("Timeout waiting for text embedding to complete")
    text_embedding_vector = None
else:
    # Calculate final timing if we completed successfully
    if 'end_time' in locals():
        print("Text embedding completed successfully")
        print(f"📊 SUMMARY - Total time from Bedrock call to output.json: {total_duration:.2f} seconds")
    else:
        print("Text embedding completed successfully (timing data not available)")

🕐 Starting Bedrock async invoke at: 2025-07-19 17:19:56.088
Error starting text embedding async invoke: An error occurred (ValidationException) when calling the StartAsyncInvoke operation: Invalid S3 credentials
Extracted text UID: nm5yeko0s7zt
Waiting for text embedding to complete...
Error checking for response: An error occurred (NoSuchBucket) when calling the ListObjectsV2 operation: The specified bucket does not exist
Error checking for response: An error occurred (NoSuchBucket) when calling the ListObjectsV2 operation: The specified bucket does not exist
Error checking for response: An error occurred (NoSuchBucket) when calling the ListObjectsV2 operation: The specified bucket does not exist
Error checking for response: An error occurred (NoSuchBucket) when calling the ListObjectsV2 operation: The specified bucket does not exist
Error checking for response: An error occurred (NoSuchBucket) when calling the ListObjectsV2 operation: The specified bucket does not exist
Error checkin

KeyboardInterrupt: 

In [25]:
# Section 3 - Step 2 - Query S3 Vectors for topk nearest neighbors

if text_embedding_vector is not None:
    try:
        # Query the vector index with the text embedding
        print("🔍 Starting S3 vector query...")
        start_time = time.time()
        
        response = s3_vectors.query_vectors(
            vectorBucketName=vector_bucket_name,
            indexName=indexName,
            topK=top_k,
            queryVector={
                'float32': text_embedding_vector
            },
            returnMetadata=True,
            returnDistance=True
        )
        
        end_time = time.time()
        query_duration_ms = (end_time - start_time) * 1000
        
        print(f"⏱️  S3 Vector Query completed in {query_duration_ms:.1f} milliseconds")
        print(f"Found {len(response['vectors'])} similar vectors:")
        print("-" * 60)
        
        # Display results
        for i, vector in enumerate(response['vectors'], 1):
            print(f"Result {i}:")
            print(f"  Key: {vector['key']}")
            print(f"  Distance: {vector.get('distance', 'N/A')}")
            
            # Display metadata if available
            if 'metadata' in vector and vector['metadata']:
                print(f"  Metadata:")
                for key, value in vector['metadata'].items():
                    print(f"    {key}: {value}")
            
            print("-" * 40)
            
    except Exception as e:
        print(f"❌ Error querying S3 vectors: {e}")
        print(f"Error type: {type(e).__name__}")
else:
    print("❌ No text embedding vector available for querying")

print("\n✅ Vector query complete!")

🔍 Starting S3 vector query...
⏱️  S3 Vector Query completed in 511.6 milliseconds
Found 4 similar vectors:
------------------------------------------------------------
Result 1:
  Key: tears-of-steel.mov_segment_0196_467.40-472.20_visual-text
  Distance: 0.6072227954864502
  Metadata:
    end_sec: 472.1999816894531
    embedding_option: visual-text
    video_uri: s3://test-video-demo-081df67e/tears-of-steel.mov
    start_sec: 467.3999938964844
----------------------------------------
Result 2:
  Key: tears-of-steel.mov_segment_0214_573.90-580.50_visual-text
  Distance: 0.6126875877380371
  Metadata:
    video_uri: s3://test-video-demo-081df67e/tears-of-steel.mov
    end_sec: 580.5
    embedding_option: visual-text
    start_sec: 573.9000244140625
----------------------------------------
Result 3:
  Key: tears-of-steel.mov_segment_0197_472.20-478.10_visual-text
  Distance: 0.6301818490028381
  Metadata:
    video_uri: s3://test-video-demo-081df67e/tears-of-steel.mov
    start_sec: 472.2

In [28]:
# Section 3 - Step 3 - Play video back at the start of the time code from the nearest neighbor

if 'response' in locals() and response and 'vectors' in response and len(response['vectors']) > 0:
    # Extract start_sec from the top result's metadata
    top_result = response['vectors'][0]
    if 'metadata' in top_result and top_result['metadata'] and 'start_sec' in top_result['metadata']:
        start_time = float(top_result['metadata']['start_sec'])
        print(f"The search term {text_search_phrase} is found at {start_time} seconds, this is the segment that matches the search term the best")
    else:
        # Fallback: try to parse from the key if it contains timing info
        start_time = 0
        print("No start_sec found in metadata, using default start time of 0")
else:
    start_time = 0
    print("No query results available, using default start time of 0")

print(f"Starting video at {start_time} seconds")

HTML(f"""
<video id="mymovie" width="640" controls>
  <source src="{test_video_name}" type="video/mp4">
  Your browser does not support the video tag.
</video>
<script>
  var video = document.getElementById('mymovie');
  video.addEventListener('loadedmetadata', function() {{
    video.currentTime = {start_time};
  }}, false);
</script>
""")

The search term a person fighting a robot is found at 467.3999938964844 seconds, this is the segment that matches the search term the best
Starting video at 467.3999938964844 seconds


##### Section 4 - Clean up of S3 bucket and S3 Vectors bucket and index, and local file
###### Step 1 - Clean up S3 bucket and S3 Vector resources

In [27]:
# Section 4 - Step 1 - Clean up S3 bucket and S3 Vector resources

if os.path.exists(test_video_name):
    os.remove(test_video_name)
    print(f"Deleted {test_video_name}")


# First empty the S3 bucket
try:
    # List all objects in the bucket
    s3_response = s3.list_objects_v2(Bucket=bucket_name)
    
    if 'Contents' in s3_response:
        # Delete all objects
        objects_to_delete = [{'Key': obj['Key']} for obj in s3_response['Contents']]
        s3.delete_objects(
            Bucket=bucket_name,
            Delete={'Objects': objects_to_delete}
        )
        print(f"Deleted {len(objects_to_delete)} objects from bucket {bucket_name}")
    else:
        print(f"Bucket {bucket_name} is already empty")
    
    # Delete the S3 bucket
    s3.delete_bucket(Bucket=bucket_name)
    print(f"Deleted S3 bucket: {bucket_name}")
    
except Exception as e:
    print(f"Error cleaning up S3 bucket: {e}")

# Delete the vector index
try:
    s3_vectors_index_delete_response = s3_vectors.delete_index(
        vectorBucketName=vector_bucket_name,
        indexName=indexName
    )
    
    # Check response status and details
    print(f"Delete index response: {s3_vectors_index_delete_response}")
    
    # Check if response indicates success
    if hasattr(s3_vectors_index_delete_response, 'get'):
        status = s3_vectors_index_delete_response.get('ResponseMetadata', {}).get('HTTPStatusCode')
        if status:
            print(f"HTTP Status Code: {status}")
            if status not in [200, 204]:
                print(f"Warning: Unexpected status code {status}")
except Exception as e:
    print(f"Error deleting vector index: {e}")

# Delete the vector bucket
try:
    s3_vectors_bucket_delete_response = s3_vectors.delete_vector_bucket(
        vectorBucketName=vector_bucket_name
    )
    print(f"Deleted vector bucket: {vector_bucket_name}")
except Exception as e:
    print(f"Error deleting vector bucket: {e}")

print("Cleanup completed")


Deleted tears-of-steel.mov
Deleted 7 objects from bucket test-video-demo-081df67e
Deleted S3 bucket: test-video-demo-081df67e
Delete index response: {'ResponseMetadata': {'RequestId': '2773009a-55e5-4b69-97a6-499296cc4929', 'HostId': '', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Sun, 20 Jul 2025 00:17:23 GMT', 'content-type': 'application/json', 'content-length': '2', 'connection': 'keep-alive', 'x-amz-request-id': '2773009a-55e5-4b69-97a6-499296cc4929', 'access-control-allow-origin': '*', 'vary': 'origin, access-control-request-method, access-control-request-headers', 'access-control-expose-headers': '*'}, 'RetryAttempts': 0}}
HTTP Status Code: 200
Deleted vector bucket: test-vectors-292e771f
Cleanup completed
